# OCT BO-DBA Denoising Study — Exploratory Data Analysis

This notebook loads the study's results CSV and reproduces the core analyses:
recovery rate by denoising method, clean-accuracy cost, ensemble strategies,
error correlation between denoisers, perturbation-magnitude effects, and
per-class breakdowns.

**Expected input:** a CSV with (at minimum) these columns —
`Run/Image ID`, `File Name`, `True Label`, `Processing`, `Prediction`, `Confidence`,
`CNV`, `DME`, `DRUSEN`, `NORMAL`, `Correct?`, `Attacked?`, `Noise Generator`, `Seed`,
`Query Budget`, `Queries Used`, `Linf`, `L2`, `SSIM`, `PSNR`.

Missing columns are handled gracefully — cells that depend on a missing column
will print a note and skip, rather than error out.

**`Processing` values expected:** `Original`, `Adversarial`, and for each denoiser
(`Gaussian`, `Median`, `Bilateral`, `Non-Local Means`): `<Method>-Attack` and `<Method>-Clean`.


## Key Definitions

A quick reference for the metrics and terms used throughout this notebook.

**Attack metrics**
- **L2 Distance**: the Euclidean (straight-line) distance between the adversarial image and the original clean image, measured across all pixels. Smaller L2 = a subtler, harder-to-notice perturbation. This is the attack-strength metric BO-DBA is specifically optimized to minimize.
- **L∞ (Linf) Distance**: the *maximum single-pixel* change between the adversarial and original image, rather than the overall distance across all pixels. Two images can have the same L2 but very different L∞ — L∞ tells you about the single worst-case pixel change, while L2 tells you about the overall change.
- **Queries Used**: how many times the attack algorithm had to query the classifier (submit an image, get a label back) before successfully fooling it. Lower = more query-efficient attack — this is BO-DBA's core selling point versus older decision-based attacks.
- **Query Budget**: the maximum number of queries the attack was allowed to use before giving up.
- **Adversarial Confidence**: the classifier's confidence score in the (wrong) label it was tricked into predicting on the attacked image. A confidence close to 50% means the attack barely tipped the image over the decision boundary; a high confidence means the attack produced a decisive, confident misclassification.

**Image similarity metrics** (used to compare a processed image back to the original clean image)
- **SSIM (Structural Similarity Index)**: scored 0–1, measures how structurally similar two images are (edges, textures, contrast) — closer to how a human eye perceives similarity. 1 = identical.
- **PSNR (Peak Signal-to-Noise Ratio)**: measured in decibels (dB), a purely pixel-value-based distortion measure. Higher = more similar. Unlike SSIM, it doesn't account for perceptual/structural similarity, so the two metrics can disagree — a useful cross-check.

**Study-specific metrics**
- **Recovery Rate**: among images the attack successfully fooled, the % that a given denoiser restored to the *correct* label. This is the core "does the defense work" metric.
- **Clean Accuracy Maintained**: applying a denoiser to a *clean, never-attacked* image, the % that still classify correctly afterward. This is the "cost of defense" control — a denoiser that recovers attacked images but also breaks clean ones isn't actually useful.
- **Any-Method Recovery / Ceiling**: the % of images where *at least one* of the denoisers recovered the correct label, even if the others failed. This is a theoretical upper bound, not something a real system gets for free — see Section 8.
- **Majority Vote / Highest Confidence**: two simple ways to combine the denoisers' outputs into a single decision instead of relying on just one method. See Section 8 for how each works and why they perform differently.

**Processing labels used throughout the data**
- `Original` — clean image, no attack, no denoising.
- `Adversarial` — attacked image, no denoising.
- `<Method>-Attack` — a denoiser applied to the *attacked* image (tests recovery).
- `<Method>-Clean` — the same denoiser applied to the *original* image (tests cost to clean accuracy).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.max_columns", None)


## 1. Load the data

**Goal:** get the results data into a pandas DataFrame so the rest of the notebook can analyze it.

Update `CSV_PATH` to point to your results file.

In [ ]:
CSV_PATH = "OCT_Results.xlsx"  # <-- update this (also accepts .xlsx)

if CSV_PATH.lower().endswith((".xlsx", ".xls")):
    df = pd.read_excel(CSV_PATH)
else:
    df = pd.read_csv(CSV_PATH)

print(f"Loaded {len(df)} rows, {df.shape[1]} columns")
df.head()

In [ ]:
# Normalize column names we rely on (adjust here if your CSV uses different headers)
COL_MAP = {
    # canonical_name: [possible header variants to check]
}
print("Columns found:", list(df.columns))


## 2. Basic sanity checks

**Goal:** catch data problems early — wrong image counts, class imbalance, duplicate images — before drawing any conclusions from downstream analysis.

In [ ]:
METHODS = ["Gaussian", "Median", "Bilateral", "Non-Local Means", "Average", "Morphological Opening"]

n_images = df["Run"].nunique() if "Run" in df.columns else df["File Name"].nunique()
print(f"Unique images: {n_images}")

if "Processing" in df.columns:
    print()
    print("Rows per Processing type:")
    print(df["Processing"].value_counts())

if "True Label" in df.columns:
    print()
    orig = df[df["Processing"] == "Original"] if "Processing" in df.columns else df
    print("Class balance (True Label, Original rows):")
    print(orig["True Label"].value_counts())

# Check for duplicate images
if "File Name" in df.columns and "Processing" in df.columns:
    orig_rows = df[df["Processing"] == "Original"]
    dupes = orig_rows["File Name"].value_counts()
    dupes = dupes[dupes > 1]
    if len(dupes):
        print()
        print("WARNING: duplicate File Name entries among Original rows:")
        print(dupes)


## 3. Build a wide, per-image lookup

**Goal:** reshape the data so every image's full story (clean, attacked, all denoised versions) sits in one row, which is what every analysis below actually needs.

Pivots the long-format data into one row per image, with each Processing
condition's prediction/correctness/confidence as its own set of columns.
This makes downstream comparisons (recovery, ensembles, correlations) much easier.


In [ ]:
def build_wide(df):
    wide = {}
    key_col = "Run" if "Run" in df.columns else "File Name"
    for _, row in df.iterrows():
        run = row[key_col]
        proc = row["Processing"]
        if run not in wide:
            wide[run] = {"true_label": row.get("True Label")}
        wide[run][f"{proc}_pred"] = row.get("Prediction")
        wide[run][f"{proc}_conf"] = row.get("Confidence")
        wide[run][f"{proc}_correct"] = row.get("Correct?")
        if proc == "Adversarial":
            wide[run]["l2"] = row.get("L2")
            wide[run]["linf"] = row.get("Linf")
            wide[run]["queries_used"] = row.get("Query Used")
            wide[run]["query_budget"] = row.get("Query Budget")
            wide[run]["noise_generator"] = row.get("Noise Generator") if "Noise Generator" in df.columns else row.get("BODBA Noise Generator")
    return pd.DataFrame.from_dict(wide, orient="index")

wide = build_wide(df)
wide.index.name = "Run"
print(f"Built wide table: {wide.shape[0]} images x {wide.shape[1]} columns")
wide.head()


## 4. Recovery Rate by denoising method

**Goal:** answer the study's central question — does each denoiser actually restore correct classification after an attack, and which one does it best?

Among images where the attack succeeded (Adversarial prediction != True Label), what fraction did each denoiser restore to the correct label?

In [ ]:
attacked_mask = wide["Adversarial_correct"] == "N" if "Adversarial_correct" in wide.columns else pd.Series(True, index=wide.index)
attacked = wide[attacked_mask]
print(f"Attack success: {len(attacked)} / {len(wide)} images ({len(attacked)/len(wide)*100:.1f}%)")

recovery_rates = {}
for m in METHODS:
    col = f"{m}-Attack_correct"
    if col in attacked.columns:
        recovery_rates[m] = (attacked[col] == "Y").mean()

recovery_df = pd.Series(recovery_rates, name="Recovery Rate").sort_values(ascending=False)
recovery_df


In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
(recovery_df * 100).plot(kind="bar", ax=ax, color="#4C72B0")
ax.set_ylabel("Recovery Rate (%)")
ax.set_title("Recovery Rate by Denoising Method")
ax.set_ylim(0, 100)
for i, v in enumerate(recovery_df * 100):
    ax.text(i, v + 1, f"{v:.1f}%", ha="center")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


## 5. Clean-Accuracy Cost by denoising method

**Goal:** check whether each denoiser's benefit comes at a cost — does it damage accuracy on images that were never attacked in the first place?

Does applying each denoiser to a clean, correctly-classified image reduce accuracy?

In [ ]:
clean_cost = {}
for m in METHODS:
    col = f"{m}-Clean_correct"
    if col in wide.columns:
        clean_cost[m] = (wide[col] == "Y").mean()

clean_cost_df = pd.Series(clean_cost, name="Clean Accuracy Maintained").sort_values(ascending=False)
clean_cost_df


In [ ]:
fig, ax = plt.subplots(figsize=(8,4.5))
x = np.arange(len(METHODS))
w = 0.35
rec_vals = [recovery_rates.get(m, np.nan) * 100 for m in METHODS]
cost_vals = [clean_cost.get(m, np.nan) * 100 for m in METHODS]

ax.bar(x - w/2, rec_vals, w, label="Recovery Rate", color="#4C72B0")
ax.bar(x + w/2, cost_vals, w, label="Clean Accuracy Maintained", color="#55A868")
ax.set_xticks(x)
ax.set_xticklabels(METHODS, rotation=15)
ax.set_ylabel("%")
ax.set_ylim(0, 100)
ax.set_title("Recovery Rate vs. Clean-Accuracy Cost, by Method")
ax.legend()
plt.tight_layout()
plt.show()


## 6. Recovery vs. Perturbation Magnitude (L2 distance)

**Goal:** test the intuitive assumption that a stronger (larger L2) attack should be harder to reverse — and see whether the data actually supports it.

Does a stronger attack (larger L2) make recovery harder?

In [ ]:
if "l2" in attacked.columns:
    bins = [0, 10, 50, 100, np.inf]
    labels = ["<10", "10-50", "50-100", ">100"]
    attacked = attacked.copy()
    attacked["l2_bin"] = pd.cut(attacked["l2"], bins=bins, labels=labels, right=False)

    any_recovery = attacked[[f"{m}-Attack_correct" for m in METHODS if f'{m}-Attack_correct' in attacked.columns]].eq("Y").any(axis=1)
    attacked["any_recovered"] = any_recovery

    bin_summary = attacked.groupby("l2_bin").agg(
        n=("l2", "count"),
        any_method_recovery=("any_recovered", "mean"),
    )
    for m in METHODS:
        col = f"{m}-Attack_correct"
        if col in attacked.columns:
            bin_summary[f"{m}_recovery"] = attacked.groupby("l2_bin")[col].apply(lambda s: (s=="Y").mean())

    display(bin_summary)

    fig, ax = plt.subplots(figsize=(8,4.5))
    bin_summary[[c for c in bin_summary.columns if "recovery" in c]].plot(kind="bar", ax=ax)
    ax.set_ylabel("Recovery Rate")
    ax.set_title("Recovery Rate by Perturbation Magnitude (L2 Distance)")
    ax.legend(bbox_to_anchor=(1.02,1), loc="upper left")
    plt.tight_layout()
    plt.show()
else:
    print("L2 column not found — skipping this analysis.")


## 7. Recovery Rate by True Class

**Goal:** check whether recovery is uniform across diagnostic classes, or whether some conditions (e.g. DRUSEN) are systematically harder to defend than others.

In [ ]:
class_recovery = attacked.groupby("true_label").apply(
    lambda g: pd.Series({m: (g[f"{m}-Attack_correct"]=="Y").mean() for m in METHODS if f"{m}-Attack_correct" in g.columns})
)
display(class_recovery)

fig, ax = plt.subplots(figsize=(8,4.5))
class_recovery.plot(kind="bar", ax=ax)
ax.set_ylabel("Recovery Rate")
ax.set_title("Recovery Rate by True Diagnostic Class")
ax.legend(bbox_to_anchor=(1.02,1), loc="upper left")
plt.tight_layout()
plt.show()


### 7b. Adversarial Confidence by True Class

How confidently-wrong were the successful attacks, per class? A lower average confidence suggests the attack barely tipped the image over — i.e., that class sits closer to the classifier's decision boundary at baseline, which helps explain why some classes (e.g. a class with weak baseline recall) are both easier to attack and harder for denoising to recover.

In [ ]:
adv_conf_by_class = attacked.groupby("true_label")["Adversarial_conf"].agg(["mean", "std", "count"])
adv_conf_by_class = adv_conf_by_class.rename(columns={"mean": "Mean Adv. Confidence", "std": "Std Dev", "count": "N"})
adv_conf_by_class = adv_conf_by_class.sort_values("Mean Adv. Confidence")
display(adv_conf_by_class)

fig, ax = plt.subplots(figsize=(7,4))
adv_conf_by_class["Mean Adv. Confidence"].plot(kind="bar", ax=ax, color="#C44E52")
ax.set_ylabel("Mean Adversarial Confidence")
ax.set_title("How Confidently-Wrong Were Successful Attacks, by True Class?")
ax.axhline(0.5, color="gray", linestyle="--", linewidth=1, label="50% (coin-flip threshold)")
ax.legend()
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

print()
print("Lower mean confidence = attacks on this class landed closer to a 50/50 call,")
print("suggesting these images sit closer to the decision boundary at baseline.")

## 8. Ensemble strategies

**Goal:** test whether combining multiple denoisers' outputs (rather than picking just one) can beat the best single method.

Two simple, zero-extra-compute ensembles built from the four denoisers'
existing predictions:
- **Majority Vote**: the class predicted by the most denoisers wins (ties noted separately).
- **Highest Confidence**: use whichever denoiser's prediction has the highest confidence score.


In [ ]:
def majority_vote_row(row):
    preds = [row.get(f"{m}-Attack_pred") for m in METHODS if pd.notna(row.get(f"{m}-Attack_pred"))]
    if not preds:
        return None, True
    vc = pd.Series(preds).value_counts()
    top = vc.max()
    winners = vc[vc == top].index.tolist()
    if len(winners) > 1:
        return None, True  # tie
    return winners[0], False

def highest_conf_row(row):
    best_m, best_c = None, -1
    for m in METHODS:
        c = row.get(f"{m}-Attack_conf")
        if pd.notna(c) and c > best_c:
            best_c = c
            best_m = m
    if best_m is None:
        return None
    return row.get(f"{best_m}-Attack_pred")

maj_results = attacked.apply(majority_vote_row, axis=1)
attacked = attacked.copy()
attacked["majority_pred"] = maj_results.apply(lambda x: x[0])
attacked["majority_tie"] = maj_results.apply(lambda x: x[1])
attacked["highest_conf_pred"] = attacked.apply(highest_conf_row, axis=1)

maj_acc = (attacked.loc[~attacked["majority_tie"], "majority_pred"] == attacked.loc[~attacked["majority_tie"], "true_label"]).mean()
maj_tie_rate = attacked["majority_tie"].mean()
conf_acc = (attacked["highest_conf_pred"] == attacked["true_label"]).mean()

print(f"Majority Vote accuracy (excl. ties): {maj_acc*100:.1f}%   |   Tie rate: {maj_tie_rate*100:.1f}%")
print(f"Highest Confidence accuracy: {conf_acc*100:.1f}%")
print()
print("Best single method:", recovery_df.idxmax(), f"({recovery_df.max()*100:.1f}%)")

# Any-method ceiling
any_method = attacked[[f"{m}-Attack_correct" for m in METHODS if f'{m}-Attack_correct' in attacked.columns]].eq("Y").any(axis=1)
print(f"'Any method' ceiling: {any_method.mean()*100:.1f}%")


In [ ]:
fig, ax = plt.subplots(figsize=(8,4.5))
labels = list(recovery_df.index) + ["Majority Vote", "Highest Confidence", "Any Method (ceiling)"]
values = list(recovery_df.values*100) + [maj_acc*100, conf_acc*100, any_method.mean()*100]
colors = ["#4C72B0"]*len(recovery_df) + ["#C44E52", "#55A868", "#8172B2"]

ax.bar(labels, values, color=colors)
ax.set_ylabel("Accuracy / Recovery Rate (%)")
ax.set_title("Single Methods vs. Ensemble Strategies")
ax.set_ylim(0,100)
plt.xticks(rotation=25, ha="right")
for i,v in enumerate(values):
    ax.text(i, v+1, f"{v:.1f}%", ha="center", fontsize=9)
plt.tight_layout()
plt.show()


### 8b. Best pairwise ensemble search

**Goal:** test every possible pair of methods with Highest-Confidence selection, to check
whether a smaller, well-chosen ensemble can beat both the best single method and the
full combined ensemble. Low error-correlation (Section 9) alone doesn't guarantee a good
pair — a pair needs to combine reasonable independence *with* individual strength.


In [ ]:
from itertools import combinations

def highest_conf_acc(methods_subset, df=attacked):
    correct = 0
    n = len(df)
    for _, row in df.iterrows():
        confs, preds = {}, {}
        for m in methods_subset:
            c = row.get(f"{m}-Attack_conf")
            p = row.get(f"{m}-Attack_pred")
            if pd.notna(c):
                confs[m] = c
                preds[m] = p
        if not confs:
            continue
        best_m = max(confs, key=lambda m: confs[m])
        if preds[best_m] == row["true_label"]:
            correct += 1
    return correct, correct / n * 100

def any_ceiling(methods_subset, df=attacked):
    cols = [f"{m}-Attack_correct" for m in methods_subset if f"{m}-Attack_correct" in df.columns]
    n_recovered = df[cols].eq("Y").any(axis=1).sum()
    return n_recovered, n_recovered / len(df) * 100

pair_results = []
for m1, m2 in combinations(METHODS, 2):
    c, acc = highest_conf_acc([m1, m2])
    _, ceiling = any_ceiling([m1, m2])
    pair_results.append({"Pair": f"{m1} + {m2}", "Correct": c, "Accuracy (%)": round(acc,1), "Any-of-2 Ceiling (%)": round(ceiling,1)})

pair_df = pd.DataFrame(pair_results).sort_values("Accuracy (%)", ascending=False).reset_index(drop=True)
display(pair_df)

best_pair = pair_df.iloc[0]
single_best_acc = recovery_df.max() * 100
full_acc = conf_acc * 100

print()
print("Best pair:", best_pair["Pair"], "—", str(best_pair["Accuracy (%)"]) + "%")
print(f"Best single method: {recovery_df.idxmax()} — {single_best_acc:.1f}%")
print(f"Full {len(METHODS)}-method ensemble: {full_acc:.1f}%")

if best_pair["Accuracy (%)"] > full_acc and best_pair["Accuracy (%)"] > single_best_acc:
    print()
    print("--> The best pair beats BOTH the best single method AND the full ensemble.")
    print("    This suggests ensemble quality depends on choosing strong, complementary")
    print("    methods -- not simply combining as many methods as possible.")


## 9. Error correlation between denoisers

**Goal:** explain *why* the ensemble strategies in Section 8 perform the way they do — if denoisers fail on the same images rather than different ones, combining them won't help as much as you'd expect.

Do the four methods fail on the *same* images more than you'd expect by chance?
This explains why naive voting can underperform the best single method.


In [ ]:
error_flags = pd.DataFrame({
    m: (attacked[f"{m}-Attack_correct"] != "Y") for m in METHODS if f"{m}-Attack_correct" in attacked.columns
})

corr = error_flags.astype(int).corr()
display(corr.round(3))

fig, ax = plt.subplots(figsize=(5.5,4.5))
sns.heatmap(corr, annot=True, cmap="Reds", vmin=0, vmax=1, ax=ax, square=True)
ax.set_title("Correlation Between Denoisers' Errors")
plt.tight_layout()
plt.show()


In [ ]:
# Observed vs. expected co-failure counts (assuming independence)
n = len(error_flags)
print(f"{'Pair':<28}{'Observed':<10}{'Expected (indep.)':<20}{'Excess %':<10}")
for m1, m2 in combinations(error_flags.columns, 2):
    obs = (error_flags[m1] & error_flags[m2]).sum()
    p1 = error_flags[m1].mean()
    p2 = error_flags[m2].mean()
    exp = p1 * p2 * n
    excess = (obs/exp - 1) * 100 if exp > 0 else np.nan
    print(f"{m1+' & '+m2:<28}{obs:<10}{exp:<20.1f}{excess:<10.1f}")


## 10. Distribution of #methods that recovered each image

**Goal:** see whether recovery is roughly all-or-nothing per image, or whether different methods genuinely catch different cases — which tells you how much value a smarter ensemble could realistically add.

In [ ]:
n_recovered = error_flags.apply(lambda row: (~row).sum(), axis=1)  # count of methods that got it right
dist = n_recovered.value_counts().sort_index()
print("How many of the 4 methods recovered each image:")
print(dist)
print()
for k in range(1,5):
    pct = (n_recovered >= k).mean() * 100
    print(f"At least {k} method(s) recovered: {pct:.1f}%")

fig, ax = plt.subplots(figsize=(6,4))
dist.plot(kind="bar", ax=ax, color="#4C72B0")
ax.set_xlabel("# of denoisers that recovered the image")
ax.set_ylabel("# of images")
ax.set_title("Distribution of Recovery Across Methods")
plt.tight_layout()
plt.show()


## 11. SSIM / PSNR by condition

**Goal:** get a visual-quality view of what each denoiser is doing to the images, independent of whether the classifier ultimately gets the label right.

Structural/pixel similarity to the original clean image, for each denoised condition.

In [ ]:
ssim_cols = [c for c in df.columns if c.upper()=="SSIM"]
psnr_cols = [c for c in df.columns if c.upper()=="PSNR"]

if ssim_cols and "Processing" in df.columns:
    plot_df = df[df["Processing"].isin([f"{m}-Attack" for m in METHODS] + [f"{m}-Clean" for m in METHODS])].copy()
    plot_df["Method"] = plot_df["Processing"].str.replace("-Attack","").str.replace("-Clean","")
    plot_df["Condition"] = plot_df["Processing"].apply(lambda x: "Attack" if "Attack" in x else "Clean")

    fig, axes = plt.subplots(1, 2, figsize=(12,4.5))
    sns.boxplot(data=plot_df, x="Method", y=ssim_cols[0], hue="Condition", ax=axes[0])
    axes[0].set_title("SSIM by Method and Condition")
    axes[0].tick_params(axis='x', rotation=15)

    if psnr_cols:
        # handle PSNR possibly stored as text like "35.9 dB"
        plot_df["_psnr_num"] = pd.to_numeric(plot_df[psnr_cols[0]].astype(str).str.replace("dB","").str.strip(), errors="coerce")
        sns.boxplot(data=plot_df, x="Method", y="_psnr_num", hue="Condition", ax=axes[1])
        axes[1].set_title("PSNR by Method and Condition")
        axes[1].set_ylabel("PSNR (dB)")
        axes[1].tick_params(axis='x', rotation=15)

    plt.tight_layout()
    plt.show()
else:
    print("SSIM/PSNR columns not found — skipping.")


## 12. Summary stats printout

**Goal:** pull every key number from the notebook into one place, so results can be quoted or shared without re-running everything.

In [ ]:
print("="*60)
print("SUMMARY")
print("="*60)
print(f"Total images: {len(wide)}")
print(f"Attack success rate: {len(attacked)/len(wide)*100:.1f}%")
print()
print("Recovery Rate by method:")
for m, v in recovery_df.items():
    print(f"  {m:<20} {v*100:5.1f}%")
print()
print("Clean Accuracy Maintained by method:")
for m, v in clean_cost_df.items():
    print(f"  {m:<20} {v*100:5.1f}%")
print()
print(f"Best single method: {recovery_df.idxmax()} ({recovery_df.max()*100:.1f}%)")
print(f"Majority Vote: {maj_acc*100:.1f}% (tie rate {maj_tie_rate*100:.1f}%)")
print(f"Highest Confidence: {conf_acc*100:.1f}%")
print(f"Any-method ceiling: {any_method.mean()*100:.1f}%")
print()
print("Most correlated error pair:")
max_pair, max_excess = None, -np.inf
for m1, m2 in combinations(error_flags.columns, 2):
    obs = (error_flags[m1] & error_flags[m2]).sum()
    exp = error_flags[m1].mean() * error_flags[m2].mean() * n
    excess = (obs/exp - 1) if exp > 0 else -np.inf
    if excess > max_excess:
        max_excess, max_pair = excess, (m1, m2)
print(f"  {max_pair[0]} & {max_pair[1]}  (+{max_excess*100:.0f}% more co-failures than expected)")
